# Landslide FS demo

This notebook runs the complete FS workflow from `example/shp/demo.shp`, then maps the final FS raster and key intermediate rasters.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "fs_tools").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fs_tools import calculate_fs_from_shp

shp_path = PROJECT_ROOT / "example" / "shp" / "demo.shp"
output_dir = PROJECT_ROOT / "example" / "output_demo"

shp_path, output_dir

In [ ]:
result = calculate_fs_from_shp(
    shp_path,
    output_dir,
    download=True,
    reuse_existing=True,
    b=30,
    cv=0.3,
)

result

For a quick offline preview with the already committed example rasters, change `output_dir` to `PROJECT_ROOT / "example" / "output"` and keep `reuse_existing=True`.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt


def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        nodata = src.nodata
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    if nodata is not None:
        arr[arr == nodata] = np.nan
    arr[~np.isfinite(arr)] = np.nan
    return arr, extent


rasters = {
    "DEM": output_dir / "dem.tif",
    "Slope": output_dir / "temp" / "slope.tif",
    "Soil depth": output_dir / "soil_depth.tif",
    "Initial hw": output_dir / "hw.tif",
    "FS": output_dir / "fs.tif",
    "FS std": output_dir / "fs_std.tif",
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for ax, (title, path) in zip(axes.ravel(), rasters.items()):
    arr, extent = read_raster(path)
    image = ax.imshow(arr, extent=extent, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(image, ax=ax, shrink=0.8)

plt.show()

In [ ]:
fs, _ = read_raster(output_dir / "fs.tif")
valid_fs = fs[np.isfinite(fs)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid_fs, bins=40, color="#2563eb", edgecolor="white")
ax.axvline(1.0, color="#dc2626", linestyle="--", linewidth=2, label="FS = 1")
ax.set_title("FS distribution")
ax.set_xlabel("Factor of safety")
ax.set_ylabel("Pixel count")
ax.legend()
plt.show()

print(f"FS min: {np.nanmin(valid_fs):.3f}")
print(f"FS mean: {np.nanmean(valid_fs):.3f}")
print(f"FS max: {np.nanmax(valid_fs):.3f}")